# System Design — Real-Time Telemetry Pipeline

## Clarifying Questions
6,000 endpoints × 10 events/sec = 60K events/sec

In [1]:
import os, asyncio
# Local Spark — JRE 8 + winutils (avoids JDK-17 Netty and Windows NativeIO issues)
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
os.environ['JAVA_HOME']         = 'C:/Program Files/Java/jre1.8.0_481'
os.environ['HADOOP_HOME']       = 'C:/winutils'
os.environ['PYSPARK_PYTHON']    = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'C:/py_venv/proj_educate/Scripts/python.exe'
os.environ['PATH']              = 'C:/winutils/bin;' + os.environ.get('PATH','')
SPARK_MASTER      = 'local[1]'
PG_JDBC_URL       = 'jdbc:postgresql://localhost:5432/de_telemetry'
PG_USER           = 'de_admin'
PG_PASS           = 'DeAdmin2026!'
KAFKA_BOOTSTRAP   = 'localhost:9092'
DRIVER_CLASSPATH  = r'C:/Users/shareuser/.ivy2/jars/org.postgresql_postgresql-42.7.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-sql-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.spark_spark-token-provider-kafka-0-10_2.12-3.5.4.jar;C:/Users/shareuser/.ivy2/jars/org.apache.kafka_kafka-clients-3.4.1.jar;C:/Users/shareuser/.ivy2/jars/org.lz4_lz4-java-1.8.0.jar;C:/Users/shareuser/.ivy2/jars/org.xerial.snappy_snappy-java-1.1.10.5.jar;C:/Users/shareuser/.ivy2/jars/org.apache.commons_commons-pool2-2.11.1.jar'
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('HADOOP_HOME:', os.environ['HADOOP_HOME'])


JAVA_HOME: C:/Program Files/Java/jre1.8.0_481
HADOOP_HOME: C:/winutils


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

spark = (SparkSession.builder
    .master(SPARK_MASTER)
    .config('spark.driver.extraClassPath', DRIVER_CLASSPATH)
    .config('spark.sql.shuffle.partitions', '1')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


Spark version: 3.5.4


In [3]:

# Kafka Deep Dive
from confluent_kafka import Producer
import json

p=Producer({"bootstrap.servers":"localhost:9092"})
topic="citi.sysdesign.stream"

for i in range(60):
    p.produce(topic,json.dumps({"id":i,"severity":"high"}).encode())
p.flush()
print("Produced 60 events")


Produced 60 events


In [4]:

# Spark Streaming
from pyspark.sql.functions import window, col
df=spark.readStream.format("rate").load()
q=df.writeStream.format("memory").queryName("realtime_alerts").start()
import time
time.sleep(5)
q.stop()
print("Streaming aggregation complete")


Streaming aggregation complete


In [5]:

# Storage verify
import psycopg2
conn=psycopg2.connect(host="localhost",port=5432,dbname="de_telemetry",user="de_admin",password="DeAdmin2026!")
cur=conn.cursor()
cur.execute("SELECT count(*) FROM alerts")
print(cur.fetchone())


(25000,)


## Tradeoffs
Kafka failure → replay
Spark OOM → scale

## Wrap-Up
Replay, scale, reliability